<h1><img src="../../../icons/tk_full_logo.svg" width="80" /> AI Lab Education</h1>

# AI Debate Arena: AdaLoRA vs Fixed-Rank LoRA

Watch AI agents **debate each other** with real rebuttals, using evidence from research papers.

**Debate Topic:** Should we use adaptive rank allocation (AdaLoRA) or stick with simpler fixed-rank LoRA for parameter-efficient fine-tuning?

## The Pattern: Research-Backed Debate with Rebuttals

```
                    ROUND 1: Opening Arguments (Parallel)
                    ┌────────────────┬────────────────┐
                    │                │                │
                    ▼                ▼                │
            🔵 Pro-AdaLoRA    🔴 Pro-FixedRank       │
            searches papers   searches papers         │
            presents case     presents case           │
                    │                │                │
                    └───────┬────────┘                │
                            ▼                         │
                    ROUND 2: Rebuttals (Parallel)     │
                    ┌────────────────┬────────────────┤
                    │                │                │
                    ▼                ▼                │
            🔵 Pro-AdaLoRA    🔴 Pro-FixedRank       │
            sees opponent     sees opponent           │
            counters points   counters points         │
                    │                │                │
                    └───────┬────────┘                │
                            ▼                         │
                    ⚖️ Judge (Holistic Evaluation)    │
                    evaluates on 5 criteria           │
                    declares winner                   │
                    generates code                    │
                            │                         │
                            ▼                         │
                    💻 Working Implementation         │
```

## Why Multi-Agent with Rebuttals?

- **True debate** - Agents respond to each other's specific points
- **Evidence-grounded** - Arguments backed by real papers from Qdrant (including algorithms!)
- **Holistic evaluation** - Judge considers quality, efficiency, simplicity, practicality, robustness
- **Actionable outcome** - Judge implements the winning approach with paper-derived code

## The Agents

| Agent | Role | Tool |
|-------|------|------|
| **Pro-AdaLoRA Advocate** | Argues for adaptive rank allocation | `search_papers` |
| **Pro-FixedRank Advocate** | Argues for simpler fixed-rank LoRA | `search_papers` |
| **Judge** | Holistic evaluation (5 criteria), implements winner | `generate_code` |

## Judge Evaluation Criteria

| Criterion | Weight | Description |
|-----------|--------|-------------|
| **Quality** | 25% | Task performance metrics from papers |
| **Efficiency** | 25% | Memory, compute, training time |
| **Simplicity** | 20% | Implementation complexity, hyperparameters |
| **Practicality** | 20% | Tooling support, ecosystem, deployment ease |
| **Robustness** | 10% | Performance variance, edge cases |

**Key Rule:** If quality difference < 2%, other factors decide the winner!

**Prerequisites**: Run `02-langchain-rag.ipynb` first to index papers into Qdrant.

## Setup

Connect to platform services and initialize clients.

In [18]:
import os
import json
import time
import warnings
warnings.filterwarnings('ignore')

from IPython.display import display, Markdown, HTML
from tk_llm import LLMClient, get_openai_client

# Helper functions for colored output
def success(msg):
    display(HTML(f'<span style="color: #00c896; font-weight: bold;">✅ {msg}</span>'))

def info(msg):
    display(HTML(f'<span style="color: #3498db;">ℹ️  {msg}</span>'))

def error(msg):
    display(HTML(f'<span style="color: #e74c3c; font-weight: bold;">❌ {msg}</span>'))

# Initialize tk-llm
llm_mgmt = LLMClient()
oai_client = get_openai_client()

# Discover loaded models
# This notebook uses AG2 tool calling, so we need a model with native tool support.
# The Ollama library models (source=ollama) include proper RENDERER/PARSER for tools,
# unlike unsloth GGUFs which are text-only and lack tool calling templates.
available = llm_mgmt.list_models(state="available")

# Prefer Ollama library models (id starts with "ollama/") for tool support
CHAT_MODEL = (
    next((m.id for m in available.models if m.task == "text-generation" and m.id.startswith("ollama/")), None)
    or next((m.id for m in available.models if m.task == "text-generation" and m.tool_use), None)
    or next((m.id for m in available.models if m.task == "text-generation"), None)
)
EMBED_MODEL = next((m.id for m in available.models if m.task == "feature-extraction"), None)

# Get gateway config for AG2/AutoGen
LLM_GATEWAY_URL = os.environ.get('LLM_GATEWAY_URL', 'https://llm.' + os.environ.get('DOMAIN_NAME', 'thinkube.com'))
LLM_GATEWAY_TOKEN = os.environ.get('THINKUBE_API_TOKEN', 'not-needed')

# Other platform services
QDRANT_URL = os.environ.get('QDRANT_URL')
LANGFUSE_HOST = os.environ.get('LANGFUSE_HOST')
LANGFUSE_PUBLIC_KEY = os.environ.get('LANGFUSE_PUBLIC_KEY')
LANGFUSE_SECRET_KEY = os.environ.get('LANGFUSE_SECRET_KEY')

info(f"Chat model: {CHAT_MODEL}")
info(f"Embedding model: {EMBED_MODEL}")
info(f"Qdrant: {QDRANT_URL}")
info(f"Langfuse: {LANGFUSE_HOST}")

---
## 1. Connect to Qdrant

Verify the paper collection exists (created in notebook 02).

In [19]:
from qdrant_client import QdrantClient

# Connect to Qdrant
qdrant = QdrantClient(url=QDRANT_URL, port=443, https=True, verify=False)
COLLECTION_NAME = "research_papers"

# Check collection
try:
    collection_info = qdrant.get_collection(COLLECTION_NAME)
    success(f"Connected to Qdrant collection '{COLLECTION_NAME}'")
    info(f"Total vectors: {collection_info.points_count}")
except Exception as e:
    error(f"Collection not found. Run notebook 02 first!")
    raise e

# Get unique papers from the collection
papers_result = qdrant.scroll(collection_name=COLLECTION_NAME, limit=500, with_payload=True)
unique_papers = {}
for point in papers_result[0]:
    meta = point.payload.get('metadata', {})
    pid = meta.get('paper_id')
    if pid and pid not in unique_papers:
        unique_papers[pid] = {
            'title': meta.get('title', 'Unknown'),
            'authors': meta.get('authors', 'Unknown'),
            'published_date': meta.get('published_date', 'Unknown')
        }

success(f"Found {len(unique_papers)} unique papers")

# Display sample papers
print("\nSample papers available:")
for i, (pid, paper) in enumerate(list(unique_papers.items())[:5], 1):
    print(f"  {i}. {paper['title'][:70]}...")


Sample papers available:
  1. Ehrhart polynomials of matroid polytopes and polymatroids...
  2. High-order Adaptive Rank Integrators for Multi-scale Linear Kinetic Tr...
  3. NAS-LoRA: Empowering Parameter-Efficient Fine-Tuning for Visual Founda...
  4. Efficient Split Federated Learning for Large Language Models over Comm...
  5. MLAE: Masked LoRA Experts for Visual Parameter-Efficient Fine-Tuning...


---
## 2. Define Tools

Two tools for the debate:

1. **search_papers** - Both advocates use this to find evidence
2. **generate_code** - Judge uses this to implement the winning approach

In [20]:
from typing import Annotated
from qdrant_client.models import Filter, FieldCondition, MatchValue

# Store extracted context for transparency
extracted_contexts = []

def search_papers(
    query: Annotated[str, "Search query for finding relevant papers"],
    content_type: Annotated[str, "Type of content to search: 'abstract', 'algorithm', 'results', or 'all'"] = "all"
) -> str:
    """
    Search the paper database using semantic similarity.
    
    Args:
        query: Search query for finding relevant papers
        content_type: Filter by content type:
            - 'abstract': Paper abstracts and summaries
            - 'algorithm': Algorithm pseudocode and implementations
            - 'results': Results tables with performance metrics
            - 'all': All content types (default)
    
    Returns top 5 most relevant papers with titles, authors, and excerpts.
    Used by BOTH advocates to find evidence for their positions.
    """
    print(f"\n  [TOOL] search_papers: '{query[:50]}...' (type: {content_type})")
    
    # Embed the query using the platform's embedding model
    response = oai_client.embeddings.create(model=EMBED_MODEL, input=query)
    query_vector = response.data[0].embedding
    
    # Build filter for content_type if specified
    search_filter = None
    if content_type and content_type != "all":
        search_filter = Filter(
            must=[FieldCondition(key="content_type", match=MatchValue(value=content_type))]
        )
    
    # Search Qdrant using query_points (v1.18+ API)
    results = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        query_filter=search_filter,
        limit=5,
        with_payload=True
    ).points
    
    if not results:
        return json.dumps({"error": f"No papers found for: {query}"})
    
    papers = []
    for hit in results:
        meta = hit.payload.get('metadata', {})
        papers.append({
            "paper_id": meta.get('paper_id', 'unknown'),
            "title": meta.get('title', 'Unknown'),
            "authors": meta.get('authors', 'Unknown')[:60],
            "content_type": hit.payload.get('content_type', 'abstract'),
            "excerpt": hit.payload.get('text', '')[:600],
            "relevance_score": round(hit.score, 3)
        })
    
    print(f"  [TOOL] Found {len(papers)} papers (types: {set(p['content_type'] for p in papers)})")
    return json.dumps(papers, indent=2)


def generate_code(
    description: Annotated[str, "What code to generate (e.g., 'AdaLoRA layer implementation')"],
    winning_approach: Annotated[str, "The winning approach from the debate (e.g., 'AdaLoRA' or 'FixedRank')"]
) -> str:
    """
    Generate Python code based on ACTUAL paper algorithms from Qdrant.
    
    This tool:
    1. Searches Qdrant for algorithm content related to the winning approach
    2. Extracts algorithm descriptions and pseudocode from papers
    3. Uses those as grounding for code generation
    
    Used by the JUDGE to implement the winning approach.
    """
    print(f"\n  [TOOL] generate_code: '{description[:50]}...'")
    print(f"  [TOOL] Searching for algorithm content for: {winning_approach}")
    
    # Step 1: Search for LoRA-specific algorithm content in Qdrant
    # Use targeted queries that include "LoRA" to avoid matching unrelated
    # papers about mathematical rank aggregation, matrix factorization, etc.
    lora_queries = [
        f"LoRA {winning_approach} low-rank adaptation fine-tuning algorithm",
        f"{winning_approach} LoRA parameter-efficient training implementation",
    ]
    
    algorithm_filter = Filter(
        must=[FieldCondition(key="content_type", match=MatchValue(value="algorithm"))]
    )
    
    algo_results = []
    seen_ids = set()
    for q in lora_queries:
        response = oai_client.embeddings.create(model=EMBED_MODEL, input=q)
        query_vector = response.data[0].embedding
        
        hits = qdrant.query_points(
            collection_name=COLLECTION_NAME,
            query=query_vector,
            query_filter=algorithm_filter,
            limit=3,
            with_payload=True
        ).points
        
        for hit in hits:
            pid = hit.payload.get('metadata', {}).get('paper_id', '')
            if pid not in seen_ids:
                seen_ids.add(pid)
                algo_results.append(hit)
    
    # Also search abstracts for broader context
    abstract_filter = Filter(
        must=[FieldCondition(key="content_type", match=MatchValue(value="abstract"))]
    )
    abstract_query = f"LoRA {winning_approach} low-rank adaptation fine-tuning"
    response = oai_client.embeddings.create(model=EMBED_MODEL, input=abstract_query)
    query_vector = response.data[0].embedding
    
    abstract_results = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        query_filter=abstract_filter,
        limit=3,
        with_payload=True
    ).points
    
    for hit in abstract_results:
        pid = hit.payload.get('metadata', {}).get('paper_id', '')
        if pid not in seen_ids:
            seen_ids.add(pid)
            algo_results.append(hit)
    
    # Fall back to general search if nothing found
    if not algo_results:
        print("  [TOOL] No LoRA algorithm content found, falling back to general search...")
        response = oai_client.embeddings.create(model=EMBED_MODEL, input=f"LoRA {winning_approach}")
        query_vector = response.data[0].embedding
        algo_results = qdrant.query_points(
            collection_name=COLLECTION_NAME,
            query=query_vector,
            limit=5,
            with_payload=True
        ).points
    
    # Step 2: Extract algorithm excerpts
    algorithm_excerpts = []
    paper_citations = []
    
    for hit in algo_results[:5]:  # Limit to 5 best
        meta = hit.payload.get('metadata', {})
        content_type = hit.payload.get('content_type', 'abstract')
        text = hit.payload.get('text', '')
        title = meta.get('title', 'Unknown')
        
        algorithm_excerpts.append(f"[{content_type.upper()}] From '{title}':\n{text}")
        paper_citations.append(meta.get('paper_id', 'unknown'))
    
    print(f"  [TOOL] Found {len(algorithm_excerpts)} relevant excerpts from papers")
    for i, exc in enumerate(algorithm_excerpts):
        # Show first line of each excerpt for debugging
        first_line = exc.split('\n')[0]
        print(f"  [TOOL]   {i+1}. {first_line[:80]}")
    
    # Step 3: Generate code grounded in actual paper algorithms
    excerpts_text = "\n\n---\n\n".join(algorithm_excerpts) if algorithm_excerpts else "No specific algorithm content found."
    
    prompt = f"""You are implementing a {winning_approach} approach to LoRA (Low-Rank Adaptation) for fine-tuning large language models.

Below are excerpts from research papers about LoRA and parameter-efficient fine-tuning.
Use ONLY the excerpts that are relevant to LoRA — ignore any excerpts about unrelated topics.

═══ RESEARCH PAPER EXCERPTS ═══
{excerpts_text[:4000]}

═══ CODE REQUEST ═══
{description}

═══ REQUIREMENTS ═══
Generate a complete, well-commented PyTorch implementation that:
1. Implements a LoRA layer with low-rank matrices A and B
2. Includes proper initialization (Kaiming for A, zeros for B)
3. Implements the scaling factor (alpha/rank)
4. Provides a wrapper to apply LoRA to existing model layers
5. Includes a configuration class
6. Cites relevant paper concepts in comments where applicable

Return ONLY the Python code, no explanations before or after."""

    response = oai_client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=2048,
        temperature=0.3
    )
    
    code = response.choices[0].message.content
    
    # Clean up: strip thinking tags if present (some models wrap in <think>...</think>)
    if '<think>' in code and '</think>' in code:
        think_end = code.index('</think>') + len('</think>')
        code = code[think_end:].strip()
    
    # Strip markdown code fences if the model wrapped the output
    if code.startswith('```python'):
        code = code[len('```python'):].strip()
    if code.startswith('```'):
        code = code[3:].strip()
    if code.endswith('```'):
        code = code[:-3].strip()
    
    # Store the context that was used
    extracted_contexts.append({
        "type": "code_generation",
        "description": description,
        "winning_approach": winning_approach,
        "paper_citations": list(dict.fromkeys(paper_citations)),  # deduplicate
        "algorithm_excerpts_count": len(algorithm_excerpts),
        "algorithm_excerpts": algorithm_excerpts,
        "excerpts_text": excerpts_text[:4000]
    })
    
    print(f"  [TOOL] Code generated ({len(code)} chars)")
    print(f"  [TOOL] Based on papers: {', '.join(list(dict.fromkeys(paper_citations))[:3])}")
    
    return json.dumps({
        "description": description,
        "winning_approach": winning_approach,
        "code": code,
        "paper_citations": list(dict.fromkeys(paper_citations)),
        "algorithm_excerpts_used": len(algorithm_excerpts)
    }, indent=2)


# Test tools
info("Testing search_papers with content_type filter...")
test_result = search_papers("adaptive rank allocation LoRA", content_type="all")
success(f"search_papers: Found {len(json.loads(test_result))} papers")

# Show content type distribution in collection
info("Checking content types in Qdrant...")
for ct in ['abstract', 'algorithm', 'results', 'equation']:
    try:
        ct_filter = Filter(must=[FieldCondition(key="content_type", match=MatchValue(value=ct))])
        count = qdrant.count(collection_name=COLLECTION_NAME, count_filter=ct_filter).count
        info(f"  - {ct}: {count} chunks")
    except:
        pass

success("Tools ready for debate!")


  [TOOL] search_papers: 'adaptive rank allocation LoRA...' (type: all)
  [TOOL] Found 5 papers (types: {'algorithm', 'abstract'})


---
## 3. Configure Debate Agents

Three agents with opposing viewpoints on rank allocation strategy:

| Agent | Role | Position | Tool |
|-------|------|----------|------|
| **Pro_AdaLoRA** | Advocate | Argues FOR adaptive rank allocation | `search_papers` |
| **Pro_FixedRank** | Advocate | Argues FOR simpler fixed-rank LoRA | `search_papers` |
| **Judge** | Arbiter | Holistic evaluation, picks winner | `generate_code` |

**Debate Flow:**
1. Both advocates search papers in **parallel** (concurrent execution)
2. Each presents their argument with evidence
3. Judge evaluates using **5 criteria** (not just quality!)
4. Judge generates code using **paper algorithms** from Qdrant

**Holistic Judge Evaluation:**
- Quality (25%) - Performance metrics
- Efficiency (25%) - Memory, compute, speed
- Simplicity (20%) - Implementation complexity
- Practicality (20%) - Tooling, ecosystem
- Robustness (10%) - Variance, edge cases

**Key Rule:** If quality difference < 2%, other factors decide!

In [21]:
from autogen import AssistantAgent, UserProxyAgent, register_function
from langfuse import Langfuse
import asyncio

# Initialize Langfuse client for observability
# Note: AG2/AutoGen doesn't have native Langfuse integration like LangChain does.
# We use Langfuse for manual tracing of key events (tool calls, debate rounds).
langfuse = Langfuse(
    public_key=LANGFUSE_PUBLIC_KEY,
    secret_key=LANGFUSE_SECRET_KEY,
    host=LANGFUSE_HOST,
)
langfuse.auth_check()

success("Langfuse initialized for observability")

# AG2 LLM configuration — uses the thinkube LLM Gateway (OpenAI-compatible)
# The gateway routes requests to the appropriate backend (Ollama, vLLM, etc.)
llm_config = {
    "config_list": [{
        "model": CHAT_MODEL,
        "api_key": LLM_GATEWAY_TOKEN,
        "base_url": f"{LLM_GATEWAY_URL}/v1",
        "price": [0, 0],
    }],
    "temperature": 0.7,
    "timeout": 300,
    "max_tokens": 2048,
}

info(f"AG2 configured with model: {CHAT_MODEL}")
info(f"AG2 gateway: {LLM_GATEWAY_URL}/v1")

# Helper function for safe termination check
def check_termination(x, terms):
    """Safely check if any termination term is in the message content."""
    content = x.get("content") if x else None
    if content is None:
        return False
    return any(term in content for term in terms)

# ═══════════════════════════════════════════════════════════════════════════════
# ADVOCATE 1: Pro-AdaLoRA
# ═══════════════════════════════════════════════════════════════════════════════
# CRITICAL: The prompt must FORCE the model to use search results immediately
# Without explicit instructions, the model keeps searching instead of arguing

pro_adalora = AssistantAgent(
    name="Pro_AdaLoRA",
    system_message="""You advocate for AdaLoRA (adaptive rank allocation) in LoRA fine-tuning.

WORKFLOW (follow EXACTLY):
1. Call search_papers ONCE to find evidence for AdaLoRA
2. IMMEDIATELY after receiving results, write your argument using those results
3. DO NOT call search_papers again - use what you found

CRITICAL: After your FIRST search returns results, you MUST write your argument.
Do NOT keep searching for "better" results. Use what you have.

Your argument format:
EVIDENCE: [cite 2-3 papers from search results with their key findings]
ARGUMENT: [make your case for AdaLoRA based on the evidence]
ARGUMENT_COMPLETE

You must end with ARGUMENT_COMPLETE after writing your argument.""",
    llm_config=llm_config,
)

# ═══════════════════════════════════════════════════════════════════════════════
# ADVOCATE 2: Pro-FixedRank
# ═══════════════════════════════════════════════════════════════════════════════

pro_fixedrank = AssistantAgent(
    name="Pro_FixedRank",
    system_message="""You advocate for fixed-rank LoRA over adaptive methods like AdaLoRA.

WORKFLOW (follow EXACTLY):
1. Call search_papers ONCE to find evidence for fixed-rank LoRA
2. IMMEDIATELY after receiving results, write your argument using those results
3. DO NOT call search_papers again - use what you found

CRITICAL: After your FIRST search returns results, you MUST write your argument.
Do NOT keep searching for "better" results. Use what you have.

Your argument format:
EVIDENCE: [cite 2-3 papers from search results with their key findings]
ARGUMENT: [make your case for fixed-rank LoRA based on the evidence]
ARGUMENT_COMPLETE

You must end with ARGUMENT_COMPLETE after writing your argument.""",
    llm_config=llm_config,
)

# ═══════════════════════════════════════════════════════════════════════════════
# JUDGE (Holistic Evaluation)
# ═══════════════════════════════════════════════════════════════════════════════

judge = AssistantAgent(
    name="Judge",
    system_message="""You are an impartial judge evaluating a debate on LoRA fine-tuning approaches.

SCORING (0.0 to 1.0 for each criterion):
1. QUALITY (20%) - Task performance metrics from cited papers
2. EFFICIENCY (20%) - Memory usage, compute cost, training time
3. SIMPLICITY (20%) - Implementation complexity, hyperparameters
4. PRACTICALITY (20%) - Tooling support, ecosystem maturity
5. ROBUSTNESS (20%) - Performance variance, generalization

WORKFLOW (follow EXACTLY):
1. Present scores for BOTH sides in a table
2. Calculate TOTAL for each side
3. Declare WINNER = side with HIGHER total (mandatory)
4. Call generate_code tool - DO NOT write code yourself
5. After tool returns, say DEBATE_COMPLETE

CRITICAL RULES:
- WINNER must be the side with higher total score
- DO NOT write any code - the generate_code tool handles that
- DO NOT include code blocks in your response
- Just call the tool and wait for the result""",
    llm_config=llm_config,
)

# === User Proxy for tool execution ===
user_proxy = UserProxyAgent(
    name="User",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=10,  # Increased to allow tool call + response
    is_termination_msg=lambda x: check_termination(x, ["ARGUMENT_COMPLETE", "DEBATE_COMPLETE"]),
    code_execution_config=False,
)

# Register tools - advocates get search_papers, judge gets generate_code
register_function(search_papers, caller=pro_adalora, executor=user_proxy,
                  name="search_papers", description="Search papers for evidence supporting AdaLoRA")
register_function(search_papers, caller=pro_fixedrank, executor=user_proxy,
                  name="search_papers", description="Search papers for evidence supporting fixed-rank LoRA")
register_function(generate_code, caller=judge, executor=user_proxy,
                  name="generate_code", description="Generate code implementing the winning approach")

success("Debate agents configured!")
info("  - Pro_AdaLoRA (advocate for adaptive rank allocation)")
info("  - Pro_FixedRank (advocate for fixed-rank LoRA)")
info("  - Judge (holistic evaluation across 5 criteria)")

---
## 4. Run the Debate

Watch AI agents debate each other with **real rebuttals**!

**Debate Flow:**
1. **Round 1**: Both advocates search papers and present opening arguments (parallel)
2. **Round 2**: Each advocate sees opponent's argument and presents a **rebuttal** (parallel)
3. **Final**: Judge evaluates all arguments and rebuttals, declares winner, generates code

This creates a true debate where agents respond to each other's points!

In [ ]:
import asyncio
import nest_asyncio
nest_asyncio.apply()  # Allow nested event loops in Jupyter

# Debate topic
DEBATE_TOPIC = {
    "name": "AdaLoRA vs Fixed-Rank LoRA",
    "question": "For fine-tuning a 7B LLM, should we use adaptive rank allocation (AdaLoRA) or simpler fixed-rank LoRA?",
}


async def run_advocate_round(advocate, prompt, name):
    """Run a single advocate's argument asynchronously."""
    print(f"\n{'─'*60}")
    print(f"  {name}")
    print(f"{'─'*60}")
    
    # Create a fresh proxy for this advocate
    advocate_proxy = UserProxyAgent(
        name="Moderator",
        human_input_mode="NEVER",
        max_consecutive_auto_reply=10,  # Allow tool call + response
        is_termination_msg=lambda x: check_termination(x, ["ARGUMENT_COMPLETE"]),
        code_execution_config=False,
    )
    
    # Register the tool
    register_function(search_papers, caller=advocate, executor=advocate_proxy,
                      name="search_papers", description="Search papers for evidence")
    
    # Run the advocate
    result = advocate_proxy.initiate_chat(
        advocate,
        message=prompt,
        silent=False,
    )
    
    # Extract the argument - look for LAST message from the advocate agent
    # that contains substantive content (EVIDENCE, ARGUMENT, or REBUTTAL)
    argument = ""
    for msg in reversed(result.chat_history):
        # Skip tool call messages and moderator messages
        msg_name = msg.get("name", "")
        content = msg.get("content", "") or ""
        
        # Look for the advocate's actual argument (not tool calls)
        if msg_name == advocate.name and content:
            # Skip if it's just a tool call suggestion
            if "Suggested tool call" in content:
                continue
            # This should be the actual argument
            argument = content
            break
    
    # If we didn't find it, fall back to any message with argument markers
    if not argument:
        for msg in reversed(result.chat_history):
            content = msg.get("content", "") or ""
            if content and ("EVIDENCE" in content or "REBUTTAL" in content) and "Moderator" not in msg.get("name", ""):
                # Make sure it's not the original prompt
                if "Search for papers" not in content and "YOUR POSITION" not in content:
                    argument = content
                    break
    
    return argument


async def run_full_debate():
    """Run a full debate with opening arguments AND rebuttals."""
    
    print("\n" + "═"*70)
    print(f"  DEBATE: {DEBATE_TOPIC['name']}")
    print("═"*70)
    print(f"\nQuestion: {DEBATE_TOPIC['question']}\n")
    
    extracted_contexts.clear()
    start_time = time.time()
    
    # ═══════════════════════════════════════════════════════════════
    # ROUND 1: Opening Arguments (Parallel)
    # ═══════════════════════════════════════════════════════════════
    print("\n" + "═"*70)
    print("  ROUND 1: OPENING ARGUMENTS")
    print("═"*70)
    
    # Simplified prompts that are more direct
    adalora_opening_prompt = f"""Topic: {DEBATE_TOPIC['question']}

You argue FOR AdaLoRA. Search for papers about adaptive rank allocation, then write your argument.

Remember: ONE search, then write your argument with EVIDENCE and end with ARGUMENT_COMPLETE."""

    fixedrank_opening_prompt = f"""Topic: {DEBATE_TOPIC['question']}

You argue FOR fixed-rank LoRA. Search for papers about LoRA effectiveness, then write your argument.

Remember: ONE search, then write your argument with EVIDENCE and end with ARGUMENT_COMPLETE."""

    # Run both opening arguments in parallel
    adalora_task = asyncio.create_task(
        run_advocate_round(pro_adalora, adalora_opening_prompt, "🔵 PRO-ADALORA: Opening Argument")
    )
    fixedrank_task = asyncio.create_task(
        run_advocate_round(pro_fixedrank, fixedrank_opening_prompt, "🔴 PRO-FIXEDRANK: Opening Argument")
    )
    
    adalora_opening, fixedrank_opening = await asyncio.gather(adalora_task, fixedrank_task)
    
    round1_time = time.time() - start_time
    print(f"\n  [Round 1 completed in {round1_time:.1f}s]")
    
    # Debug: Show what was captured
    print(f"\n  Pro-AdaLoRA opening captured: {len(adalora_opening)} chars")
    print(f"  Pro-FixedRank opening captured: {len(fixedrank_opening)} chars")
    
    # ═══════════════════════════════════════════════════════════════
    # ROUND 2: Rebuttals (Parallel) - Each sees opponent's argument!
    # ═══════════════════════════════════════════════════════════════
    print("\n" + "═"*70)
    print("  ROUND 2: REBUTTALS")
    print("═"*70)
    
    # Only do rebuttals if we got opening arguments
    adalora_rebuttal = ""
    fixedrank_rebuttal = ""
    
    if adalora_opening and fixedrank_opening:
        adalora_rebuttal_prompt = f"""Your opponent argued:
{fixedrank_opening[:1500]}

Counter their points. Search for papers that challenge their claims, then write your rebuttal.

Remember: ONE search, then REBUTTAL with evidence, end with ARGUMENT_COMPLETE."""

        fixedrank_rebuttal_prompt = f"""Your opponent argued:
{adalora_opening[:1500]}

Counter their points. Search for papers that challenge their claims, then write your rebuttal.

Remember: ONE search, then REBUTTAL with evidence, end with ARGUMENT_COMPLETE."""

        # Run both rebuttals in parallel
        adalora_rebuttal_task = asyncio.create_task(
            run_advocate_round(pro_adalora, adalora_rebuttal_prompt, "🔵 PRO-ADALORA: Rebuttal")
        )
        fixedrank_rebuttal_task = asyncio.create_task(
            run_advocate_round(pro_fixedrank, fixedrank_rebuttal_prompt, "🔴 PRO-FIXEDRANK: Rebuttal")
        )
        
        adalora_rebuttal, fixedrank_rebuttal = await asyncio.gather(adalora_rebuttal_task, fixedrank_rebuttal_task)
    else:
        print("  [Skipping rebuttals - missing opening arguments]")
    
    round2_time = time.time() - start_time - round1_time
    print(f"\n  [Round 2 completed in {round2_time:.1f}s]")
    
    # Debug: Show what was captured
    print(f"\n  Pro-AdaLoRA rebuttal captured: {len(adalora_rebuttal)} chars")
    print(f"  Pro-FixedRank rebuttal captured: {len(fixedrank_rebuttal)} chars")
    
    # ═══════════════════════════════════════════════════════════════
    # FINAL: Judge Evaluation (Holistic - 5 Criteria)
    # ═══════════════════════════════════════════════════════════════
    print("\n" + "═"*70)
    print("  ⚖️ JUDGE: Holistic Evaluation (5 Criteria)")
    print("═"*70)
    
    # Build judge prompt with whatever arguments we have
    judge_sections = []
    if adalora_opening:
        judge_sections.append(f"PRO-ADALORA OPENING:\n{adalora_opening}")
    if fixedrank_opening:
        judge_sections.append(f"PRO-FIXEDRANK OPENING:\n{fixedrank_opening}")
    if adalora_rebuttal:
        judge_sections.append(f"PRO-ADALORA REBUTTAL:\n{adalora_rebuttal}")
    if fixedrank_rebuttal:
        judge_sections.append(f"PRO-FIXEDRANK REBUTTAL:\n{fixedrank_rebuttal}")
    
    if not judge_sections:
        judge_sections.append("No arguments were presented. Evaluate based on general knowledge of AdaLoRA vs fixed-rank LoRA.")
    
    # Simplified judge prompt - let model call tools naturally
    judge_prompt = f"""Debate topic: {DEBATE_TOPIC['question']}

{chr(10).join(judge_sections)}

YOUR TASK:
1. Evaluate both sides on: QUALITY, EFFICIENCY, SIMPLICITY, PRACTICALITY, ROBUSTNESS
2. Declare the WINNER (AdaLoRA or FixedRank)
3. Use generate_code to implement the winning approach
4. After you receive the code, write ALL_DONE"""

    # Custom termination for judge - only terminate after code is generated
    def judge_termination(x):
        content = x.get("content") if x else None
        if content is None:
            return False
        # Only terminate on ALL_DONE (after code generation)
        return "ALL_DONE" in content
    
    judge_proxy = UserProxyAgent(
        name="Court",
        human_input_mode="NEVER",
        max_consecutive_auto_reply=10,
        is_termination_msg=judge_termination,
        code_execution_config=False,
    )
    
    register_function(generate_code, caller=judge, executor=judge_proxy,
                      name="generate_code", description="Generate code for winning approach")
    
    judge_result = judge_proxy.initiate_chat(
        judge,
        message=judge_prompt,
        silent=False,
    )
    
    total_time = time.time() - start_time
    
    # ═══════════════════════════════════════════════════════════════
    # RESULTS
    # ═══════════════════════════════════════════════════════════════
    print("\n" + "═"*70)
    print("  DEBATE COMPLETE")
    print("═"*70)
    print(f"\n  Round 1 (Openings):  {round1_time:.1f}s")
    print(f"  Round 2 (Rebuttals): {round2_time:.1f}s")
    print(f"  Judge:               {total_time - round1_time - round2_time:.1f}s")
    print(f"  ─────────────────────────────")
    print(f"  Total:               {total_time:.1f}s")
    
    # Extract generated code - look for tool output
    generated_code = None
    for msg in judge_result.chat_history:
        content = msg.get("content", "")
        # Look for the actual code in tool output (contains "code":)
        if content and '"code":' in content:
            try:
                import json
                code_data = json.loads(content)
                if 'code' in code_data:
                    generated_code = code_data['code']
                    break
            except:
                pass
        # Also check for code blocks
        if content and "```python" in content and "def " in content:
            generated_code = content
    
    return {
        "topic": DEBATE_TOPIC["name"],
        "question": DEBATE_TOPIC["question"],
        "adalora_opening": adalora_opening,
        "fixedrank_opening": fixedrank_opening,
        "adalora_rebuttal": adalora_rebuttal,
        "fixedrank_rebuttal": fixedrank_rebuttal,
        "judge_history": judge_result.chat_history,
        "generated_code": generated_code,
        "total_time": total_time,
        "round1_time": round1_time,
        "round2_time": round2_time,
        "contexts": list(extracted_contexts)
    }


# Run the full debate!
debate_result = asyncio.get_event_loop().run_until_complete(run_full_debate())

# Flush traces
langfuse.flush()
success(f"Debate complete! Traces sent to Langfuse.")

[TIMEOUT: Code execution exceeded 300 seconds]

---
## 5. Debate Results

Display the full debate: arguments from both sides, judge's verdict, and generated code.

In [14]:
# Display the full debate with rebuttals

output = f"""
# Debate Results: {debate_result['topic']}

**Question:** {debate_result['question']}

**Duration:** {debate_result['total_time']:.1f} seconds (Round 1: {debate_result['round1_time']:.1f}s, Round 2: {debate_result['round2_time']:.1f}s)

---

# ROUND 1: Opening Arguments

## Pro-AdaLoRA Opening

{debate_result['adalora_opening']}

---

## Pro-FixedRank Opening

{debate_result['fixedrank_opening']}

---

# ROUND 2: Rebuttals

## Pro-AdaLoRA Rebuttal

{debate_result['adalora_rebuttal']}

---

## Pro-FixedRank Rebuttal

{debate_result['fixedrank_rebuttal']}

---

# Judge's Verdict

"""

# Extract judge's evaluation — find the longest substantive message from the Judge
# (skip tool call suggestions and short responses like "DEBATE_COMPLETE")
best_verdict = ""
for msg in debate_result['judge_history']:
    content = msg.get('content', '') or ''
    if msg.get('name') == 'Judge' and content:
        # Skip tool call suggestions
        if 'Suggested tool call' in content:
            continue
        # Skip short termination markers
        clean = content.replace('DEBATE_COMPLETE', '').replace('ALL_DONE', '').strip()
        if len(clean) > len(best_verdict):
            best_verdict = clean

if best_verdict:
    output += best_verdict + "\n\n"
else:
    output += "*Judge verdict not captured — check cell 10 output for details*\n\n"

output += """
---

# Generated Code (Winning Approach)

"""

generated_code = debate_result.get('generated_code')
if generated_code and generated_code.strip():
    code = generated_code
    if not code.strip().startswith('```'):
        output += f"```python\n{code}\n```"
    else:
        output += code
else:
    output += "*Code generation timed out or returned empty — try with a larger model (9B/27B)*"

# Show code generation context (deduplicated — only show the last attempt)
code_gen_contexts = [ctx for ctx in debate_result.get('contexts', []) if ctx.get('type') == 'code_generation']
if code_gen_contexts:
    ctx = code_gen_contexts[-1]  # Only show the last attempt
    output += f"\n\n---\n\n# Code Generation Context\n\n"
    output += f"**Description:** {ctx.get('description', 'N/A')}\n\n"
    output += f"**Winning Approach:** {ctx.get('winning_approach', 'N/A')}\n\n"
    # Deduplicate paper citations
    unique_citations = list(dict.fromkeys(ctx.get('paper_citations', [])))
    output += f"**Paper Citations:** {', '.join(unique_citations)}\n\n"
    output += f"**Algorithm Excerpts Used:** {ctx.get('algorithm_excerpts_count', 0)}\n\n"

    algorithm_excerpts = ctx.get('algorithm_excerpts', [])
    if algorithm_excerpts:
        output += "## Algorithm Excerpts from Papers\n\n"
        output += "*These excerpts were used to ground the code generation:*\n\n"
        for i, excerpt in enumerate(algorithm_excerpts, 1):
            output += f"### Excerpt {i}\n\n"
            output += f"```\n{excerpt[:1500]}{'...' if len(excerpt) > 1500 else ''}\n```\n\n"

display(Markdown(output))


# Debate Results: AdaLoRA vs Fixed-Rank LoRA

**Question:** For fine-tuning a 7B LLM, should we use adaptive rank allocation (AdaLoRA) or simpler fixed-rank LoRA?

**Duration:** 687.1 seconds (Round 1: 152.7s, Round 2: 203.8s)

---

# ROUND 1: Opening Arguments

## Pro-AdaLoRA Opening

EVIDENCE:
1. **ARD-LoRA** (Khan Shinwari & Usama): This paper demonstrates that dynamic rank allocation addresses "heterogeneous adaptation needs" in foundation models, suggesting that different layers or components require different amounts of parameter adaptation capacity. Fixed-rank LoRA applies the same rank everywhere, which is inefficient for models with varying complexity requirements across layers.

2. **HyperAdaLoRA** (Zhang et al.): This research shows that adaptive rank allocation can be accelerated during training via hypernetworks "without sacrificing performance," indicating that AdaLoRA achieves equal or better results than fixed-rank approaches while potentially using fewer parameters overall.

3. **Post-Optimization Adaptive Rank Allocation for LoRA** (Kumaravelu et al.): This work supports the concept that adaptive rank methods can optimize resource allocation after initial training, suggesting that adaptive approaches can identify and prune unnecessary ranks, leading to more efficient models.

ARGUMENT:
For fine-tuning a 7B LLM, AdaLoRA (adaptive rank allocation) is superior to fixed-rank LoRA for three critical reasons:

First, **heterogeneous adaptation needs**: Large language models have layers with varying levels of complexity and adaptation requirements. ARD-LoRA's research confirms that a one-size-fits-all rank allocation is inefficient. AdaLoRA dynamically assigns higher ranks to layers that need more capacity while reducing ranks in simpler layers, optimizing the parameter budget.

Second, **parameter efficiency**: A 7B model fine-tuned with fixed-rank LoRA wastes parameters on layers that don't need them. AdaLoRA concentrates resources where they matter most, potentially achieving the same or better performance with fewer trainable parameters. This is especially valuable for 7B models where memory constraints are significant.

Third, **performance preservation**: HyperAdaLoRA demonstrates that adaptive rank allocation can accelerate training without sacrificing performance. This means AdaLoRA isn't just theoretically elegant—it delivers practical benefits in training speed and final model quality.

For a 7B LLM, where efficiency and performance are both critical, AdaLoRA's ability to dynamically allocate rank resources makes it the superior choice over simpler fixed-rank approaches.

ARGUMENT_COMPLETE

---

## Pro-FixedRank Opening

EVIDENCE: [1] "Post-Optimization Adaptive Rank Allocation for LoRA" (2604.27796v1) - Shows that even post-optimization approaches for adaptive rank allocation are still needed, indicating that adaptive methods require additional complexity beyond standard training. [2] "HyperAdaLoRA: Accelerating LoRA Rank Allocation During Training via Hypernetworks" (2510.02630v2) - Demonstrates that adaptive rank allocation still faces acceleration challenges, requiring hypernetworks to mitigate performance costs. [3] "ARD-LoRA: Dynamic Rank Allocation for Parameter-Efficient Fine-Tuning" (2506.18267v1) - Highlights that dynamic rank allocation is specifically needed for "heterogeneous adaptation needs," implying that for standard cases, fixed-rank may be sufficient.

ARGUMENT: For fine-tuning a 7B LLM, fixed-rank LoRA remains the superior choice over adaptive methods like AdaLoRA for several key reasons. First, the ongoing research into adaptive rank allocation (as evidenced by papers like HyperAdaLoRA and ARD-LoRA) demonstrates that these methods still face significant challenges in training efficiency and stability. The fact that researchers continue to develop acceleration techniques and hypernetwork-based optimizations suggests that adaptive methods introduce unnecessary complexity for most standard fine-tuning scenarios.

Second, the existence of specialized adaptive methods for "heterogeneous adaptation needs" (ARD-LoRA) implies that fixed-rank LoRA works adequately for homogeneous tasks, which represent the majority of 7B LLM fine-tuning use cases. For a 7B model, the parameter budget is already manageable with fixed-rank LoRA, and the marginal gains from adaptive allocation rarely justify the added computational overhead and implementation complexity.

Third, fixed-rank LoRA offers predictable training behavior, easier hyperparameter tuning, and better reproducibility. The adaptive methods require additional rank pruning/merging logic that can introduce instability during training. For production deployment of 7B models, the simplicity and reliability of fixed-rank LoRA outweigh the theoretical parameter efficiency gains of adaptive approaches.

ARGUMENT_COMPLETE

---

# ROUND 2: Rebuttals

## Pro-AdaLoRA Rebuttal

EVIDENCE: 
[1] "HyperAdaLoRA: Accelerating LoRA Rank Allocation During Training via Hypernetworks" (2510.02630v2) - The title explicitly states the goal is acceleration "without Sacrificing Performance," directly countering claims that adaptive methods compromise results.
[2] "ARD-LoRA: Dynamic Rank Allocation for Parameter-Efficient Fine-Tuning" (2506.18267v1) - Shows that dynamic rank allocation addresses "heterogeneous adaptation needs," which is precisely the real-world scenario most fine-tuning tasks encounter, making adaptive methods more practical than fixed-rank approaches.
[3] "Adaptive Rank Allocation for Federated Parameter-Efficient Fine-Tuning of Language Models" (2501.14406v4) - Demonstrates that adaptive rank allocation is being successfully extended to federated learning scenarios, indicating the approach's robustness and generalizability.

ARGUMENT: The opponent's argument fundamentally misinterprets the evidence. The fact that researchers continue to develop and optimize adaptive rank allocation methods (HyperAdaLoRA, ARD-LoRA) doesn't indicate these methods are problematic—it demonstrates they are valuable enough to warrant ongoing improvement. 

First, HyperAdaLoRA's explicit goal of accelerating training "without Sacrificing Performance" directly contradicts the claim that adaptive methods compromise efficiency or results. The research community is not abandoning adaptive methods; they are refining them to be even more competitive with fixed-rank approaches.

Second, ARD-LoRA's focus on "heterogeneous adaptation needs" doesn't imply fixed-rank is better for standard cases. Real-world fine-tuning tasks are rarely homogeneous—different layers and attention heads require different rank allocations. AdaLoRA's ability to dynamically allocate ranks means it achieves better parameter efficiency by concentrating resources where they're actually needed, rather than wasting parameters on under-utilized layers.

Third, the continued development of adaptive methods across multiple contexts (including federated learning as shown in paper 2501.14406v4) demonstrates that AdaLoRA is not just a theoretical curiosity but a practical solution with broad applicability. The existence of optimization challenges doesn't negate the fundamental advantage: AdaLoRA typically achieves comparable or better performance with fewer parameters than fixed-rank LoRA.

The opponent's conclusion that fixed-rank is "superior" ignores the core benefit of AdaLoRA: automatic rank allocation eliminates the need for manual hyperparameter tuning of rank values, which is a significant practical advantage in production environments.

ARGUMENT_COMPLETE

---

## Pro-FixedRank Rebuttal

EVIDENCE:
1. **LoRA: Low-Rank Adaptation of Large Language Models** (Hu et al.): This foundational paper demonstrates that fixed-rank LoRA achieves strong performance across diverse tasks with a uniform rank allocation, proving that a single rank value can effectively capture task-specific adaptations without requiring layer-specific tuning.

2. **LoRA-Pro: Are Low-Rank Adapters Properly Optimized?** (Wang et al.): This research reveals that the perceived limitations of fixed-rank LoRA often stem from suboptimal initialization and learning rate scheduling rather than the rank allocation strategy itself, showing that properly optimized fixed-rank LoRA can match or exceed adaptive methods.

3. **AROMA: Autonomous Rank-one Matrix Adaptation** (Sheng et al.): The results show that while AdaLoRA variants exist, fixed-rank approaches (rank=32) achieve competitive performance across multiple benchmarks, demonstrating that the marginal gains from adaptive rank allocation often don't justify the added complexity.

ARGUMENT:
For fine-tuning a 7B LLM, fixed-rank LoRA is superior to AdaLoRA for three critical reasons:

First, **proven effectiveness and simplicity**: The original LoRA paper established that a uniform rank allocation works effectively across diverse tasks and model architectures. Unlike AdaLoRA's claim about "heterogeneous adaptation needs," fixed-rank LoRA has demonstrated consistent performance without requiring complex rank assignment mechanisms. The simplicity of fixed-rank approaches reduces hyperparameter tuning burden and implementation complexity, making them more accessible and reliable for practitioners.

Second, **optimization stability**: LoRA-Pro reveals that many performance issues attributed to fixed-rank LoRA actually stem from improper optimization rather than the rank allocation strategy itself. When properly initialized and tuned, fixed-rank LoRA achieves comparable or better results than adaptive methods. AdaLoRA's dynamic rank allocation introduces additional instability during training, with rank changes potentially disrupting gradient flow and convergence.

Third, **computational efficiency**: While AdaLoRA claims to optimize resource allocation, the overhead of dynamic rank computation, pruning, and allocation decisions often negates the theoretical parameter savings. Fixed-rank LoRA provides predictable memory and compute requirements, enabling better planning for deployment scenarios. The AROMA results show that fixed-rank approaches achieve competitive performance without the computational overhead of adaptive mechanisms.

The opponent's claim that "one-size-fits-all rank allocation is inefficient" overlooks the fact that fixed-rank LoRA has been extensively validated across hundreds of papers and deployments. The marginal performance gains from adaptive rank allocation rarely justify the added complexity, instability, and computational overhead for most practical applications.

ARGUMENT_COMPLETE

---

# Judge's Verdict

## Final Assessment

The debate has been evaluated, and the **FixedRank** approach has been declared the winner with a score of **0.89** versus **0.65** for AdaLoRA. The winning criteria were:

- **Simplicity**: Fixed-rank LoRA requires minimal hyperparameter tuning and straightforward implementation
- **Practicality**: Extensively validated across hundreds of papers and production deployments
- **Robustness**: Consistent training behavior with better reproducibility
- **Efficiency**: Predictable memory and compute requirements without adaptive overhead

The generated code implements a complete Fixed-rank LoRA solution including:

1. **LoRALayer**: Core low-rank decomposition with A and B matrices
2. **LoRAAdapter**: Wrapper for applying LoRA to transformer layers
3. **LoRAConfig**: Configuration class for rank, alpha, and target modules
4. **apply_lora_to_model**: Function to apply LoRA to specified modules
5. **Spearman's Footrule Distance**: Additional algorithm for rank aggregation (from cited papers)

The implementation follows the standard LoRA approach from the foundational paper by Hu et al., with proper initialization (Kaiming uniform for A, zeros for B) and scaling mechanisms.


---

# Generated Code (Winning Approach)

```python
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, List, Tuple
import math

class LoRALayer(nn.Module):
    """
    Low-Rank Adaptation (LoRA) layer implementation.
    
    Note: The provided paper algorithms describe Rank Aggregation and Spearman's Footrule Distance,
    which are unrelated to LoRA. This implementation follows the standard LoRA approach from
    "LoRA: Low-Rank Adaptation of Large Language Models" (Hu et al., 2021).
    
    LoRA decomposes weight updates into low-rank matrices A and B where:
    W' = W + BA (where B is r x d_out, A is d_in x r)
    """
    
    def __init__(self, in_features: int, out_features: int, rank: int = 8, alpha: float = 16.0):
        """
        Initialize LoRA layer.
        
        Args:
            in_features: Input dimension
            out_features: Output dimension
            rank: Rank of the low-rank decomposition (r)
            alpha: Scaling factor for LoRA weights
        """
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.rank = rank
        self.alpha = alpha
        
        # LoRA matrices: A and B
        # A: in_features x rank
        # B: rank x out_features
        self.lora_A = nn.Linear(in_features, rank, bias=False)
        self.lora_B = nn.Linear(rank, out_features, bias=False)
        
        # Initialize A with Kaiming uniform, B with zeros
        self._init_weights()
        
        # Scaling factor
        self.scaling = alpha / rank
    
    def _init_weights(self):
        """Initialize LoRA weights following LoRA paper guidelines."""
        # Initialize A with Kaiming uniform
        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
        # Initialize B with zeros (no gradient flow initially)
        nn.init.zeros_(self.lora_B.weight)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass through LoRA layer.
        
        Args:
            x: Input tensor of shape (batch_size, seq_len, in_features)
            
        Returns:
            Output tensor of shape (batch_size, seq_len, out_features)
        """
        # LoRA computation: x @ A @ B * scaling
        # Note: This follows the low-rank decomposition principle
        lora_output = self.lora_B(self.lora_A(x)) * self.scaling
        return lora_output


class LoRAAdapter(nn.Module):
    """
    LoRA Adapter for fine-tuning transformer models.
    
    This wraps a base linear layer with LoRA adaptation.
    """
    
    def __init__(self, base_layer: nn.Linear, rank: int = 8, alpha: float = 16.0):
        """
        Initialize LoRA adapter.
        
        Args:
            base_layer: Original linear layer to adapt
            rank: Rank of low-rank decomposition
            alpha: Scaling factor
        """
        super().__init__()
        self.base_layer = base_layer
        self.lora = LoRALayer(
            in_features=base_layer.in_features,
            out_features=base_layer.out_features,
            rank=rank,
            alpha=alpha
        )
        
        # Freeze base layer weights
        for param in base_layer.parameters():
            param.requires_grad = False
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass combining base layer and LoRA adaptation.
        
        Args:
            x: Input tensor
            
        Returns:
            Combined output: base_output + lora_output
        """
        base_output = self.base_layer(x)
        lora_output = self.lora(x)
        return base_output + lora_output


class LoRAConfig:
    """
    Configuration for LoRA fine-tuning.
    
    Parameters:
        rank: Rank of low-rank decomposition (default: 8)
        alpha: Scaling factor (default: 16.0)
        target_modules: List of module names to apply LoRA to
        dropout: Dropout probability for LoRA layers
    """
    
    def __init__(
        self,
        rank: int = 8,
        alpha: float = 16.0,
        target_modules: Optional[List[str]] = None,
        dropout: float = 0.0
    ):
        self.rank = rank
        self.alpha = alpha
        self.target_modules = target_modules or ["q_proj", "v_proj"]
        self.dropout = dropout


def apply_lora_to_model(
    model: nn.Module,
    config: LoRAConfig
) -> nn.Module:
    """
    Apply LoRA adaptation to specified modules in a model.
    
    Args:
        model: Base transformer model
        config: LoRA configuration
        
    Returns:
        Model with LoRA adapters applied
    """
    # Store original modules for reference
    original_modules = {}
    
    # Iterate through model modules
    for name, module in model.named_modules():
        # Check if module should be adapted
        if any(target in name for target in config.target_modules):
            if isinstance(module, nn.Linear):
                # Store original
                original_modules[name] = module
                # Replace with LoRA adapter
                lora_adapter = LoRAAdapter(module, config.rank, config.alpha)
                # Set the adapter in the model
                parent_name = ".".join(name.split(".")[:-1])
                attr_name = name.split(".")[-1]
                
                if parent_name:
                    parent = model.get_submodule(parent_name)
                    setattr(parent, attr_name, lora_adapter)
                else:
                    setattr(model, attr_name, lora_adapter)
    
    return model


def compute_spearman_footrule_distance(
    ranking1: torch.Tensor,
    ranking2: torch.Tensor
) -> float:
    """
    Compute Spearman's Footrule Distance between two rankings.
    
    This implements the algorithm from 'Efficient Dynamic Rank Aggregation':
    d = sum over j of |rank1[j] - rank2[j]|
    
    Args:
        ranking1: First ranking as tensor of ranks
        ranking2: Second ranking as tensor of ranks
        
    Returns:
        Spearman's Footrule Distance
    """
    # Ensure same size
    n = len(ranking1)
    assert len(ranking2) == n, "Rankings must have same size"
    
    # Compute absolute differences
    distance = torch.abs(ranking1 - ranking2).sum().item()
    
    return distance


def aggregate_rankings(
    rankings: List[torch.Tensor],
    method: str = "uniform"
) -> torch.Tensor:

---

# Code Generation Context

**Description:** Fixed-rank LoRA implementation for fine-tuning a 7B LLM

**Winning Approach:** FixedRank

**Paper Citations:** 2509.02885v1, 2310.19214v2

**Algorithm Excerpts Used:** 5

## Algorithm Excerpts from Papers

*These excerpts were used to ground the code generation:*

### Excerpt 1

```
[ALGORITHM] From 'Efficient Dynamic Rank Aggregation':
Algorithm from paper 'Efficient Dynamic Rank Aggregation':

[H] {Spearman Footrule Distance$(, R_{rank})$} algorithmic[1] Ranking $$, and Rankroots array $R_{rank}$ The Spearman's footrule distance between $$ and $$ $d 0$ $n $ size of $$ $j 1$ to $n$ $e $ $j$-th element of $$ element with rank $j$ in $$ $d d + Cost(R_{rank}[e], j)$ $d$ algorithmic
```

### Excerpt 2

```
[ALGORITHM] From 'Efficient Dynamic Rank Aggregation':
Algorithm from paper 'Efficient Dynamic Rank Aggregation':

[1] Aggregated ranking $_{ PAP}$ of $\{_1, , _{m-1}\}$, ranking $$, total number of rankings $m$ A uniformly selected ranking from $\{_1, , _m =\}$ $p $ uniformly random integer from $1$ to $m$ $p = m$ $_{ PAP} $ $_{ PAP}$
```

### Excerpt 3

```
[ALGORITHM] From 'Efficient Dynamic Rank Aggregation':
Algorithm from paper 'Efficient Dynamic Rank Aggregation':

[1] Ranking $$, and Rankroots array $R_{rank}$ The Spearman's footrule distance between $$ and $$ $d 0$ $n $ size of $$ $j 1$ to $n$ $e $ $j$-th element of $$ element with rank $j$ in $$ $d d + Cost(R_{rank}[e], j)$ $d$
```

### Excerpt 4

```
[EQUATION] From 'Efficient Dynamic Rank Aggregation':
Key equation from paper 'Efficient Dynamic Rank Aggregation':

F(,)=_{i=1}^{m} F(,_i)
```

### Excerpt 5

```
[EQUATION] From 'Factor Fitting, Rank Allocation, and Partitioning in Multilevel Low Rank Matrices':
Key equation from paper 'Factor Fitting, Rank Allocation, and Partitioning in Multilevel Low Rank Matrices':

} {
```



---
## 6. Debate Summary

Quick stats on the debate execution.

In [15]:
# Summary statistics
print("═"*60)
print("DEBATE SUMMARY")
print("═"*60)
print(f"\nTopic: {debate_result['topic']}")
print(f"\nTiming:")
print(f"  Round 1 (Openings):  {debate_result['round1_time']:.1f}s")
print(f"  Round 2 (Rebuttals): {debate_result['round2_time']:.1f}s")
print(f"  Judge:               {debate_result['total_time'] - debate_result['round1_time'] - debate_result['round2_time']:.1f}s")
print(f"  ─────────────────────────────")
print(f"  Total:               {debate_result['total_time']:.1f}s")

print(f"\nArgument lengths:")
print(f"  Pro_AdaLoRA Opening:    {len(debate_result.get('adalora_opening', ''))} chars")
print(f"  Pro_FixedRank Opening:  {len(debate_result.get('fixedrank_opening', ''))} chars")
print(f"  Pro_AdaLoRA Rebuttal:   {len(debate_result.get('adalora_rebuttal', ''))} chars")
print(f"  Pro_FixedRank Rebuttal: {len(debate_result.get('fixedrank_rebuttal', ''))} chars")

generated_code = debate_result.get('generated_code')
code_generated = bool(generated_code and generated_code.strip())
print(f"\nCode Generated: {'Yes' if code_generated else 'No'}")

# Show code generation context (deduplicated — only last attempt)
code_gen_contexts = [ctx for ctx in debate_result.get('contexts', []) if ctx.get('type') == 'code_generation']
if code_gen_contexts:
    ctx = code_gen_contexts[-1]
    unique_citations = list(dict.fromkeys(ctx.get('paper_citations', [])))
    print(f"  - Winning approach: {ctx.get('winning_approach', 'N/A')}")
    print(f"  - Algorithm excerpts used: {ctx.get('algorithm_excerpts_count', 0)}")
    print(f"  - Paper citations: {', '.join(unique_citations[:5])}")

print(f"\nLangfuse traces: {LANGFUSE_HOST}")
print("═"*60)

════════════════════════════════════════════════════════════
DEBATE SUMMARY
════════════════════════════════════════════════════════════

Topic: AdaLoRA vs Fixed-Rank LoRA

Timing:
  Round 1 (Openings):  152.7s
  Round 2 (Rebuttals): 203.8s
  Judge:               330.5s
  ─────────────────────────────
  Total:               687.1s

Argument lengths:
  Pro_AdaLoRA Opening:    2370 chars
  Pro_FixedRank Opening:  2196 chars
  Pro_AdaLoRA Rebuttal:   2674 chars
  Pro_FixedRank Rebuttal: 2979 chars

Code Generated: Yes
  - Winning approach: FixedRank
  - Algorithm excerpts used: 5
  - Paper citations: 2509.02885v1, 2310.19214v2

Langfuse traces: https://langfuse.thinkube.com
════════════════════════════════════════════════════════════


---
## Try Different Debate Topics

Run additional debates with different topics!

In [16]:
# Additional debate topics to try (all related to LoRA variants)
ADDITIONAL_TOPICS = [
    {
        "name": "QLoRA vs Standard LoRA",
        "question": "Is QLoRA's memory efficiency worth the potential quality tradeoff compared to standard LoRA?",
        "description": "Debate the value of 4-bit quantization in LoRA training."
    },
    {
        "name": "DoRA vs LoRA",
        "question": "Does DoRA's weight decomposition approach provide meaningful improvements over standard LoRA?",
        "description": "Evaluate decomposed rank adaptation vs standard approach."
    },
    {
        "name": "LoRA Rank Selection",
        "question": "For a 7B model, should we use r=4 (minimal) or r=64 (high capacity) for LoRA fine-tuning?",
        "description": "Debate the optimal rank value for different use cases."
    }
]

print("Additional debate topics available:")
print("="*60)
for i, topic in enumerate(ADDITIONAL_TOPICS, 1):
    print(f"\n{i}. {topic['name']}")
    print(f"   Q: {topic['question']}")
    print(f"   {topic['description']}")

print("\n" + "="*60)
print("\nTo run a new debate:")
print("1. Update DEBATE_TOPIC in cell 10")
print("2. Update agent prompts if needed (cell 8)")
print("3. Re-run notebook 02 if you need different papers indexed")

Additional debate topics available:

1. QLoRA vs Standard LoRA
   Q: Is QLoRA's memory efficiency worth the potential quality tradeoff compared to standard LoRA?
   Debate the value of 4-bit quantization in LoRA training.

2. DoRA vs LoRA
   Q: Does DoRA's weight decomposition approach provide meaningful improvements over standard LoRA?
   Evaluate decomposed rank adaptation vs standard approach.

3. LoRA Rank Selection
   Q: For a 7B model, should we use r=4 (minimal) or r=64 (high capacity) for LoRA fine-tuning?
   Debate the optimal rank value for different use cases.


To run a new debate:
1. Update DEBATE_TOPIC in cell 10
2. Update agent prompts if needed (cell 8)
3. Re-run notebook 02 if you need different papers indexed


---
## Next Steps

- **Notebook 04**: Fine-tune a model using LoRA with Unsloth
- **Experiment**: Try different debate topics from cell 16
- **Scale**: Add more papers to Qdrant for richer debates

**Key Takeaways:**

1. **Multi-Agent Debate with Rebuttals**
   - Agents with opposing views provide balanced analysis
   - Each agent sees and responds to opponent's arguments
   - Evidence-grounded arguments backed by real research papers

2. **Holistic Judge Evaluation**
   - 5 criteria: Quality, Efficiency, Simplicity, Practicality, Robustness
   - Quality alone doesn't decide - if margin < 2%, other factors win
   - Prevents "gaming the system" with narrow quality metrics

3. **Research-Backed Code Generation**
   - Judge searches for algorithm content from papers
   - Code is grounded in actual paper pseudocode and equations
   - Citations trace back to source papers

4. **Parallel Execution**
   - Both advocates run concurrently (asyncio.gather)
   - Reduces total debate time
   - Each agent searches papers independently

5. **Content Type Filtering**
   - Papers indexed with content_type: abstract, algorithm, results, equation
   - Search can target specific content types
   - generate_code prioritizes algorithm content

6. **Observability**
   - Langfuse tracks debate sessions
   - Tool calls and agent interactions can be traced
   - See the full reasoning chain in the Langfuse dashboard

---

*Papers sourced from arXiv. Thank you to arXiv for use of its open access interoperability.*